# 🛡️ Ingestion Interceptor — Complete Pipeline
### BEL Project | IIT Bhubaneswar | Detection & Prevention of Malware in RPA/Drone Feeds

---

> **Who is this notebook for?**
> Anyone involved in this project who wants to deeply understand the Ingestion Interceptor module —
> what it is, why every design decision was made, and how the code works line by line.
>
> **You do not need to read anything else.** Every concept, term, and background idea is explained here.

---

## 📋 Table of Contents

| # | Section | What you will learn |
|---|---------|--------------------|
| 0 | Background Concepts | All terms and ideas used in this notebook |
| 1 | Setup & Imports | Package structure, how to import |
| 2 | The Big Picture | Where this module sits in the 9-module pipeline |
| 3 | Configuration | `InterceptorConfig` and all tuneable parameters |
| 4 | Data Models | All dataclasses: submissions, payloads, results |
| 5 | Stage 1: Validation | Structure and constraint checking |
| 6 | Stage 2: Authentication | Device registry, reputation, signatures |
| 7 | Stage 3: Metadata Extraction | Mission context, geo, telemetry |
| 8 | Stage 4: Payload Analysis | 9-point security heuristic |
| 9 | Stage 5: Checksum Verification | Integrity and tamper detection |
| 10 | Stage 6: Artifact Management | Cataloging and storage pointers |
| 11 | Uplink Communication | Two-way commands from control center |
| 12 | Full Pipeline | `IngestionInterceptor` orchestrator class |
| 13 | End-to-End Test | 5 realistic drone scenarios |
| 14 | Quick Reference Card | One-page summary |

---

**Project context:**
- **PI:** Dr. Padmalochan Bera, IIT Bhubaneswar
- **Sponsor:** Bharat Electronics Ltd (BEL)
- **Objective:** Detection and prevention of malware and malicious file injection in RPA/Drone feeds
- **This module:** Module 1 of 9 — the very first gate every drone data packet must pass through

---
## Section 0 — Background Concepts

> **Read this first.** These terms appear throughout the notebook. They are explained here
> so you never have to look something up elsewhere.

---

### 0.1 What is an RPA/Drone Data Stream?

**RPA** = Remotely Piloted Aircraft (military/government terminology for drones).

A drone data stream is the continuous or batch flow of data that a drone sends back to
its ground control station (GCS) or a central server. This stream contains:

| Data type | Example | Typical format |
|-----------|---------|----------------|
| Video | First-person view (FPV) camera | MP4, MKV, AVI |
| Images | Still camera snapshots, thermal | JPEG, PNG, TIFF |
| Telemetry | Speed, altitude, heading, battery | JSON, CSV |
| Archives | Bundled mission data | ZIP, TAR, GZ |
| Encrypted files | Classified payload | Encrypted container |
| Metadata | Mission ID, operator, firmware | JSON fields |

In our system, each drone sends a **submission** — a JSON object containing one or more
**payloads** (files) plus contextual metadata.

---

### 0.2 What Kinds of Data Do Drones Transmit?

```
🚁 Drone in the field
 │
 ├── Video feeds          ─ FPV camera, survey recordings (largest payloads)
 ├── Still images          ─ Snapshots, thermal captures, geo-tagged photos
 ├── Telemetry data        ─ GPS, speed, heading, battery, signal strength
 ├── Mission archives      ─ Compressed bundles of multiple files
 └── Encrypted payloads    ─ Classified data, encrypted at source
```

**Key point:** Not all data is equally risky. A plain JPEG from a trusted drone is very different
from an encrypted archive with an executable extension from an unknown device.

---

### 0.3 Why is Ingestion Security Critical?

The ingestion point is the **attack surface** — the first place where external data enters
the operational network. If a compromised drone sends a malicious file, and we accept it
without checks, that file now lives inside our trusted network.

**Attack scenarios the Ingestion Interceptor prevents:**

| Attack | How it works | What we check |
|--------|-------------|---------------|
| Malware injection | Drone sends an EXE disguised as video | Extension analysis, MIME checking |
| Data exfiltration | Malicious payload phones home | Encrypted payload flagging |
| Supply chain attack | Compromised firmware sends bad data | Device authentication, reputation |
| Tampered files | Man-in-the-middle modifies files in transit | Checksum verification |
| Path traversal | Filename like `../../etc/passwd` | Filename validation |
| Spoofed identity | Attacker impersonates a trusted drone | Signature verification, device registry |
| ZIP bomb | Tiny archive that expands to TB | Container flagging (deferred to sandbox) |

---

### 0.4 What is Metadata in This Context?

Metadata = "data about data." For a drone submission, metadata includes:

- **Drone ID** — unique identifier for the aircraft
- **Mission ID / zone** — which operation this data belongs to
- **Geo-location** — latitude, longitude, altitude where the data was captured
- **Telemetry** — speed, heading, battery level, signal strength
- **Operator ID** — who is controlling the drone
- **Firmware version** — software running on the drone
- **Timestamp** — when the data was captured

The Ingestion Interceptor **extracts** this metadata and passes it downstream.
The separate **Metadata Sanitizer** (Module 5) will clean and validate it more deeply.

---

### 0.5 What are Security Flags?

Security flags are warnings that our payload analyzer raises when it detects something
unusual about a file. Each flag has a **severity level**:

| Flag | Severity | What triggers it |
|------|----------|------------------|
| `encrypted_payload` | Medium | File is marked as encrypted |
| `nested_archive` | Medium | File is a container (ZIP inside ZIP) |
| `large_binary` | Low | File exceeds 10 MB threshold |
| `suspicious_mime` | High | MIME type is executable-like |
| `executable_file` | Critical | Extension is `.exe`, `.dll`, `.bat`, etc. |
| `double_extension` | High | Filename like `photo.jpg.exe` |
| `hidden_file` | Medium | Unix hidden file (starts with `.`) |
| `zero_size_file` | Medium | 0-byte video/image (suspicious) |
| `mime_extension_mismatch` | Medium | MIME says JPEG but extension is `.png` |

Flags do **not** mean the file is definitely malicious. They tell the downstream
**Game-Theoretic Threat Estimator** to pay closer attention.

---

### 0.6 What is Device Authentication for Drones?

Just like a user logs into a website, a drone must prove its identity before we trust its data.

Our system uses a **device registry** — a database of known drones with:
- **Trust status** — is this drone trusted? (explicitly set by administrators)
- **Reputation score** — 0.0 (worst) to 1.0 (best), based on historical behavior
- **Revocation status** — has this drone been compromised and revoked?

Optionally, drones can provide a **cryptographic signature** (HMAC-SHA256 or Ed25519)
that proves the data was not tampered with in transit.

**Authentication outcomes:**
```
Device found + trusted          →  authenticated
Device found + NOT trusted      →  untrusted
Device found + revoked          →  rejected
Device NOT found + policy=flag  →  unknown (flagged, allowed through)
Device NOT found + policy=reject→  rejected (blocked)
```

---

### 0.7 What is an Uplink Command?

The Ingestion Interceptor does not only receive data — it can also receive **commands**
from the control center. This is the "uplink" channel.

| Command | Effect |
|---------|--------|
| `QUARANTINE` | Mark a specific ingestion as quarantined |
| `RELEASE` | Release a previously quarantined ingestion |
| `REVOKE_DEVICE` | Revoke a drone's trust (future submissions rejected) |
| `UPDATE_ZONE_RISK` | Change the risk score for a mission zone |
| `FORCE_RESCAN` | Request re-analysis of a previously ingested submission |
| `UPDATE_CONFIG` | Update interceptor configuration parameters |

This two-way communication is part of the BEL proposal’s requirement for
**adaptive, feedback-driven security** — the system can respond to new intelligence in real time.

---
## Section 1 — Setup & Imports
---

In [1]:
# ─── Standard library imports ────────────────────────────────────────────────────
import os, sys, json, tempfile, hashlib, time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Any
from datetime import datetime, timezone

print('✅ Standard library imports ready')

✅ Standard library imports ready


In [2]:
# ─── Import the ingestion_interceptor package ─────────────────────────────────────
# Ensure the parent directory is on the Python path
PROJECT_ROOT = os.path.dirname(os.getcwd()) if 'ingestion_interceptor' in os.listdir('.') else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from ingestion_interceptor import (
    # Main classes
    IngestionInterceptor,
    InterceptorConfig,
    ingestion_interceptor,
    # Data models
    DroneSubmission,
    PayloadEntry,
    GeoLocation,
    ArtifactRecord,
    IngestMetadata,
    IngestResult,
    # Authentication
    Authenticator,
    AuthResult,
    DeviceRegistry,
    SignatureVerifier,
    # Analysis
    analyze_payload,
    compute_payload_risk_score,
    validate_submission,
)

# Additional submodule imports for demos
from ingestion_interceptor.checksum_verifier import (
    compute_bytes_checksum,
    compute_file_checksum,
    verify_checksum,
)
from ingestion_interceptor.metadata_extractor import (
    extract_mission_context,
    extract_geo_metadata,
    extract_telemetry_summary,
    extract_additional_metadata,
)
from ingestion_interceptor.artifact_manager import (
    create_artifact_record,
    generate_artifact_id,
    generate_ingest_id,
    generate_thumbnail_ref,
    build_storage_pointer,
)
from ingestion_interceptor.uplink import (
    UplinkReceiver,
    UplinkCommandHandler,
    UplinkCommand,
    CommandType,
)
from ingestion_interceptor.payload_analyzer import (
    FLAG_SEVERITY,
    MIME_EXTENSION_MAP,
    generate_threat_notes,
)

print('✅ ingestion_interceptor package loaded successfully')

✅ ingestion_interceptor package loaded successfully


In [3]:
# ─── Package structure ──────────────────────────────────────────────────────────
print('''
📂 ingestion_interceptor/
 ├── __init__.py             Public API & re-exports
 ├── config.py               InterceptorConfig dataclass
 ├── models.py               All data models (6 dataclasses)
 ├── validator.py            Stage 1: Submission validation
 ├── authenticator.py        Stage 2: Device auth (registry + signatures)
 ├── metadata_extractor.py   Stage 3: Mission/geo/telemetry extraction
 ├── payload_analyzer.py     Stage 4: 9-point security heuristic
 ├── checksum_verifier.py    Stage 5: Integrity verification
 ├── artifact_manager.py     Stage 6: Artifact cataloging
 ├── uplink.py               Uplink commands from control center
 ├── interceptor.py          Main orchestrator (IngestionInterceptor)
 ├── run_demo.py             Standalone demo script
 └── tests/                  Unit tests
''')


📂 ingestion_interceptor/
 ├── __init__.py             Public API & re-exports
 ├── config.py               InterceptorConfig dataclass
 ├── models.py               All data models (6 dataclasses)
 ├── validator.py            Stage 1: Submission validation
 ├── authenticator.py        Stage 2: Device auth (registry + signatures)
 ├── metadata_extractor.py   Stage 3: Mission/geo/telemetry extraction
 ├── payload_analyzer.py     Stage 4: 9-point security heuristic
 ├── checksum_verifier.py    Stage 5: Integrity verification
 ├── artifact_manager.py     Stage 6: Artifact cataloging
 ├── uplink.py               Uplink commands from control center
 ├── interceptor.py          Main orchestrator (IngestionInterceptor)
 ├── run_demo.py             Standalone demo script
 └── tests/                  Unit tests



---
## Section 2 — The Big Picture

### 🔵 Theory

Our system has **9 modules**. The Ingestion Interceptor is the **very first one** —
the gateway that every piece of drone data must pass through before entering the
operational network.

**What triggers it:** An incoming drone data submission (JSON with payload references).

**What it outputs:** An `IngestResult` containing:
- `IngestMetadata` — normalized metadata, auth status, security flags, zone risk
- `ArtifactRecord[]` — one record per payload with security analysis
- `warnings` / `errors` — any issues found during processing

**What consumes its output:** The Game-Theoretic Threat Estimator (Module 2) uses the
security flags, reputation scores, and zone risk to compute a formal threat level
for each submission.

---

### 📊 Full System Architecture (9 Modules)

In [4]:
print('''
╔══════════════════════════════════════════════════════════════════════════════╗
║   BEL DRONE SECURITY PIPELINE — 9-MODULE ARCHITECTURE                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

    🚁 Drone / RPA Feed
      │
      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ① INGESTION INTERCEPTOR  ←←← THIS MODULE                           │
  │  Validate → Authenticate → Extract metadata → Analyze payloads      │
  │  → Verify checksums → Catalog artifacts                              │
  └────────────────────────────────────────────────────────────────────────┘
      │
      │  IngestResult (metadata + artifact records + security flags)
      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ② Game-Theoretic Threat Estimator                                   │
  │  Uses flags + reputation + zone risk to compute threat level           │
  └────────────────────────────────────────────────────────────────────────┘
      │
      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ③ Inspection Strategy Selector                                      │
  │  Routes to: Signature Scan / AI-ML / Sandbox based on threat level    │
  └────────────────────────────────────────────────────────────────────────┘
      │
      ├───── LOW    ─────►  ╣ Signature Scanner       ╣
      ├───── MEDIUM ─────►  ╣ AI/ML Classifier        ╣
      └───── HIGH   ─────►  ╣ Sandbox Analyzer        ╣
                               │
      ┌───────────────────────┘
      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ⑤ Metadata Sanitizer                                                │
  └────────────────────────────────────────────────────────────────────────┘
      │
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ⑥ Threat Intelligence Correlator                                    │
  └────────────────────────────────────────────────────────────────────────┘
      │
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ⑦ Response & Quarantine Manager                                     │
  └────────────────────────────────────────────────────────────────────────┘
      │
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ⑧ Logging & Feedback Loop                                           │
  └────────────────────────────────────────────────────────────────────────┘
      │
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ⑨ Security Dashboard                                                │
  └────────────────────────────────────────────────────────────────────────┘
''')


╔══════════════════════════════════════════════════════════════════════════════╗
║   BEL DRONE SECURITY PIPELINE — 9-MODULE ARCHITECTURE                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

    🚁 Drone / RPA Feed
      │
      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ① INGESTION INTERCEPTOR  ←←← THIS MODULE                           │
  │  Validate → Authenticate → Extract metadata → Analyze payloads      │
  │  → Verify checksums → Catalog artifacts                              │
  └────────────────────────────────────────────────────────────────────────┘
      │
      │  IngestResult (metadata + artifact records + security flags)
      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │  ② Game-Theoretic Threat Estimator                                   │
  │  Uses flags + reputation + zone risk to compute threat level           │
  └────────────────────────

In [5]:
# ─── Internal pipeline of the Ingestion Interceptor ─────────────────────────────
print('''
┌────────────────────────────────────────────────────────────────────────┐
│  INSIDE THE INGESTION INTERCEPTOR                                       │
└────────────────────────────────────────────────────────────────────────┘

  Raw drone JSON
      │
      ▼
  ┌──────────────────────┐
  │ Stage 1: VALIDATE    │  Check structure, types, constraints, timestamps
  └──────────┬───────────┘
             │
      ┌─────┴────────────────┐
      │ Stage 2: AUTHENTICATE    │  Device registry lookup + signature check
      └──────────┬───────────────┘
                 │
      ┌─────────┴──────────────────┐
      │ Stage 3: EXTRACT METADATA    │  Mission context, geo, telemetry
      └──────────┬───────────────────┘
                 │
      ┌─────────┴─────────────────────┐
      │ Stage 4: ANALYZE PAYLOADS       │  9-point security heuristic per file
      └──────────┬──────────────────────┘
                 │
      ┌─────────┴────────────────────────┐
      │ Stage 5: VERIFY CHECKSUMS          │  Tamper detection
      └──────────┬─────────────────────────┘
                 │
      ┌─────────┴───────────────────────────┐
      │ Stage 6: CATALOG ARTIFACTS           │  IDs, storage pointers, thumbnails
      └──────────┬────────────────────────────┘
                 │
                 ▼
           IngestResult
         (metadata + artifacts + warnings)
                 │
                 ▼
   ───►  Game-Theoretic Threat Estimator (Module 2)
''')


┌────────────────────────────────────────────────────────────────────────┐
│  INSIDE THE INGESTION INTERCEPTOR                                       │
└────────────────────────────────────────────────────────────────────────┘

  Raw drone JSON
      │
      ▼
  ┌──────────────────────┐
  │ Stage 1: VALIDATE    │  Check structure, types, constraints, timestamps
  └──────────┬───────────┘
             │
      ┌─────┴────────────────┐
      │ Stage 2: AUTHENTICATE    │  Device registry lookup + signature check
      └──────────┬───────────────┘
                 │
      ┌─────────┴──────────────────┐
      │ Stage 3: EXTRACT METADATA    │  Mission context, geo, telemetry
      └──────────┬───────────────────┘
                 │
      ┌─────────┴─────────────────────┐
      │ Stage 4: ANALYZE PAYLOADS       │  9-point security heuristic per file
      └──────────┬──────────────────────┘
                 │
      ┌─────────┴────────────────────────┐
      │ Stage 5: VERIFY CHECKSUMS         

---
## Section 3 — Configuration

### 🔵 Theory

All tuneable parameters are centralized in a single `InterceptorConfig` dataclass.
This means:
- No magic numbers scattered in code
- Easy to create different configs for different deployments (permissive vs. strict)
- Every parameter has a sensible default

The config is organized into 7 parameter groups:

| Group | Parameters | Purpose |
|-------|-----------|----------|
| Validation | `require_signature`, `max_payload_size_bytes`, `max_payloads_per_submission`, `allowed_mime_types` | Control what submissions are accepted |
| Security flags | `large_binary_threshold`, `suspicious_mime_types`, `suspicious_extensions` | Tune sensitivity of payload analysis |
| Authentication | `auth_backend`, `auth_timeout_seconds`, `unknown_device_policy` | How devices are verified |
| Checksum | `checksum_algorithm`, `verify_checksums` | Integrity verification settings |
| Storage | `storage_backend`, `storage_base_path`, `artifact_uri_prefix` | Where artifacts are stored |
| Zone risk | `default_zone_risk` | Default risk for unknown zones |
| Uplink | `uplink_enabled`, `uplink_endpoint`, `uplink_poll_interval_seconds` | Control center communication |

In [6]:
# ─── Inspect all InterceptorConfig defaults ─────────────────────────────────────
default_config = InterceptorConfig()

print('InterceptorConfig — ALL DEFAULTS')
print('=' * 65)
print(f'  require_signature          : {default_config.require_signature}')
print(f'  max_payload_size_bytes     : {default_config.max_payload_size_bytes:,} ({default_config.max_payload_size_bytes // 1_000_000} MB)')
print(f'  max_payloads_per_submission: {default_config.max_payloads_per_submission}')
print(f'  allowed_mime_types         : {len(default_config.allowed_mime_types)} types')
print(f'  large_binary_threshold     : {default_config.large_binary_threshold:,} ({default_config.large_binary_threshold // 1_000_000} MB)')
print(f'  suspicious_mime_types      : {default_config.suspicious_mime_types}')
print(f'  suspicious_extensions      : {default_config.suspicious_extensions}')
print(f'  auth_backend               : {default_config.auth_backend}')
print(f'  auth_timeout_seconds       : {default_config.auth_timeout_seconds}')
print(f'  unknown_device_policy      : {default_config.unknown_device_policy}')
print(f'  checksum_algorithm         : {default_config.checksum_algorithm}')
print(f'  verify_checksums           : {default_config.verify_checksums}')
print(f'  storage_backend            : {default_config.storage_backend}')
print(f'  storage_base_path          : {default_config.storage_base_path}')
print(f'  artifact_uri_prefix        : {default_config.artifact_uri_prefix}')
print(f'  default_zone_risk          : {default_config.default_zone_risk}')
print(f'  uplink_enabled             : {default_config.uplink_enabled}')
print(f'  uplink_endpoint            : {repr(default_config.uplink_endpoint)}')
print(f'  uplink_poll_interval_seconds: {default_config.uplink_poll_interval_seconds}')
print(f'  log_level                  : {default_config.log_level}')
print(f'  structured_logging         : {default_config.structured_logging}')

InterceptorConfig — ALL DEFAULTS
  require_signature          : False
  max_payload_size_bytes     : 500,000,000 (500 MB)
  max_payloads_per_submission: 50
  allowed_mime_types         : 14 types
  large_binary_threshold     : 10,000,000 (10 MB)
  suspicious_mime_types      : {'application/x-msdownload', 'application/octet-stream', 'application/x-executable', 'application/x-dosexec'}
  suspicious_extensions      : {'dll', 'sh', 'bat', 'ps1', 'cmd', 'exe', 'vbs', 'js', 'scr', 'com', 'msi'}
  auth_backend               : registry
  auth_timeout_seconds       : 5.0
  unknown_device_policy      : flag
  checksum_algorithm         : sha256
  verify_checksums           : True
  storage_backend            : filesystem
  storage_base_path          : drone_remote_store
  artifact_uri_prefix        : s3://forensics/artifacts
  default_zone_risk          : 0.5
  uplink_enabled             : False
  uplink_endpoint            : ''
  uplink_poll_interval_seconds: 10.0
  log_level                  :

In [7]:
# ─── DEMO: Different configs for different scenarios ───────────────────────────

# Scenario 1: Strict military deployment
strict_config = InterceptorConfig(
    require_signature=True,
    max_payload_size_bytes=100_000_000,  # 100 MB cap
    unknown_device_policy='reject',       # Reject unknown drones
    checksum_algorithm='sha256',
    storage_backend='s3',
)
print('\u2705 Strict config: signatures required, unknown devices rejected')

# Scenario 2: Permissive research/testing
permissive_config = InterceptorConfig(
    require_signature=False,
    max_payload_size_bytes=2_000_000_000,  # 2 GB cap
    unknown_device_policy='allow',          # Allow all devices
    verify_checksums=False,                 # Skip integrity checks
)
print('\u2705 Permissive config: no signatures, no checksum, all devices allowed')

# Scenario 3: BEL production deployment
production_config = InterceptorConfig(
    require_signature=False,      # Signatures optional (not all drones support it)
    unknown_device_policy='flag',  # Flag unknown devices but let them through
    verify_checksums=True,
    storage_backend='s3',
    uplink_enabled=True,
    uplink_endpoint='grpc://control.bel.internal:9090',
)
print('\u2705 Production config: balanced security with uplink enabled')

✅ Strict config: signatures required, unknown devices rejected
✅ Permissive config: no signatures, no checksum, all devices allowed
✅ Production config: balanced security with uplink enabled


---
## Section 4 — Data Models

### 🔵 Theory

The Ingestion Interceptor uses **6 dataclasses** to represent all entities.
Dataclasses give us:
- Type safety (each field has a declared type)
- Immutable-like semantics (clear structure)
- Easy serialization (`to_dict()` / `from_dict()` methods)

```
📊 Data Model Hierarchy

  DroneSubmission              (what comes IN from the drone)
  ├── drone_id, timestamp
  ├── payloads: List[PayloadEntry]
  │     ├── type, filename, mime, size_bytes
  │     ├── encryption, container
  │     └── checksum, uri
  ├── geo: GeoLocation
  │     └── lat, lon, alt
  ├── telemetry, signature
  └── mission_id, mission_zone, operator_id, firmware_version

  IngestResult                 (what goes OUT to the next module)
  ├── success: bool
  ├── ingest_metadata: IngestMetadata
  │     ├── ingest_id, drone_id, timestamp, received_at
  │     ├── auth_result, reputation, zone_risk
  │     ├── insecure_flags, notes
  │     └── num_files, total_size_bytes
  ├── artifact_records: List[ArtifactRecord]
  │     ├── artifact_id, filename, type, mime
  │     ├── security_flags, checksum_verified
  │     └── pointer_storage, thumbnail
  ├── errors: List[str]
  └── warnings: List[str]
```

In [8]:
# ─── DEMO: Constructing a DroneSubmission from raw JSON ──────────────────────────

# This is what a drone sends us: raw JSON (as a Python dict)
raw_json = {
    'drone_id': 'DRN-001',
    'timestamp': '2025-10-13T03:00:12Z',
    'mission_id': 'MSN-142',
    'mission_zone': 'zone-a',
    'geo': {'lat': 12.971598, 'lon': 77.594566, 'alt': 120},
    'payloads': [
        {'type': 'video', 'filename': 'drn001_fpv_001.mp4', 'mime': 'video/mp4',
         'size_bytes': 4500000, 'encryption': False, 'container': False,
         'checksum': 'a1b2c3d4...'},
        {'type': 'image', 'filename': 'drn001_cam_001.jpg', 'mime': 'image/jpeg',
         'size_bytes': 320000, 'encryption': False, 'container': False,
         'checksum': 'e5f6g7h8...'},
    ],
    'telemetry': {'speed': 12.5, 'heading': 145.2, 'battery': 78.4, 'signal_strength': 82.1},
    'signature': None,
    'firmware_version': 'v1.2.0',
    'operator_id': 'OP-12',
    'additional_metadata': {'camera_model': 'CAM-X1000', 'frame_rate': 30},
}

# Parse into typed dataclass
submission = DroneSubmission.from_dict(raw_json)

print('DroneSubmission parsed successfully!')
print(f'  drone_id        : {submission.drone_id}')
print(f'  timestamp       : {submission.timestamp}')
print(f'  mission_id      : {submission.mission_id}')
print(f'  mission_zone    : {submission.mission_zone}')
print(f'  geo             : lat={submission.geo.lat}, lon={submission.geo.lon}, alt={submission.geo.alt}')
print(f'  num payloads    : {len(submission.payloads)}')
for i, p in enumerate(submission.payloads):
    print(f'    [{i}] {p.filename} ({p.type}, {p.mime}, {p.size_bytes:,} bytes)')
print(f'  telemetry       : {submission.telemetry}')
print(f'  firmware_version: {submission.firmware_version}')
print(f'  operator_id     : {submission.operator_id}')
print(f'  signature       : {submission.signature}')

DroneSubmission parsed successfully!
  drone_id        : DRN-001
  timestamp       : 2025-10-13T03:00:12Z
  mission_id      : MSN-142
  mission_zone    : zone-a
  geo             : lat=12.971598, lon=77.594566, alt=120.0
  num payloads    : 2
    [0] drn001_fpv_001.mp4 (video, video/mp4, 4,500,000 bytes)
    [1] drn001_cam_001.jpg (image, image/jpeg, 320,000 bytes)
  telemetry       : {'speed': 12.5, 'heading': 145.2, 'battery': 78.4, 'signal_strength': 82.1}
  firmware_version: v1.2.0
  operator_id     : OP-12
  signature       : None


In [9]:
# ─── DEMO: Individual model classes ────────────────────────────────────────────

# GeoLocation
geo = GeoLocation(lat=12.971598, lon=77.594566, alt=120.0)
print('GeoLocation:')
print(f'  to_dict() = {geo.to_dict()}')

# GeoLocation from dict (with validation)
geo2 = GeoLocation.from_dict({'lat': 12.97, 'lon': 77.59})
print(f'  from_dict (no alt) = lat={geo2.lat}, lon={geo2.lon}, alt={geo2.alt}')

geo3 = GeoLocation.from_dict(None)
print(f'  from_dict(None)    = {geo3}')

print()

# PayloadEntry
payload = PayloadEntry(
    type='video', filename='flight_01.mp4', mime='video/mp4',
    size_bytes=4500000, encryption=False, container=False,
    checksum='a1b2c3d4...'
)
print('PayloadEntry:')
print(f'  {payload.filename} | {payload.mime} | {payload.size_bytes:,} bytes')
print(f'  encrypted={payload.encryption} | container={payload.container}')
print(f'  to_dict() keys = {list(payload.to_dict().keys())}')

GeoLocation:
  to_dict() = {'lat': 12.971598, 'lon': 77.594566, 'alt': 120.0}
  from_dict (no alt) = lat=12.97, lon=77.59, alt=0.0
  from_dict(None)    = None

PayloadEntry:
  flight_01.mp4 | video/mp4 | 4,500,000 bytes
  encrypted=False | container=False
  to_dict() keys = ['type', 'filename', 'mime', 'size_bytes', 'encryption', 'container', 'checksum']


---
## Section 5 — Stage 1: Validation

### 🔵 Theory

Validation is the **absolute first check** before any processing. It prevents:

| Attack / Error | What validation catches |
|----------------|------------------------|
| Malformed JSON | Missing required fields (`drone_id`, `timestamp`, `payloads`) |
| Empty payloads | Payloads list must be non-empty |
| Path traversal | Filenames containing `/` or `\\` (e.g., `../../etc/passwd`) |
| Oversized files | Payloads exceeding `max_payload_size_bytes` |
| Invalid timestamps | Non-ISO8601 or far-future timestamps |
| Too many payloads | Exceeding `max_payloads_per_submission` |
| MIME confusion | Unrecognized MIME types (warning, not fatal) |

The `validate_submission()` function returns `(errors, warnings)`:
- **Errors** are fatal — the submission is rejected
- **Warnings** are informational — the submission proceeds with flags

---

### 📊 Validation Flow

```
Raw JSON dict
    │
    ├── Has drone_id?          ── NO  → error: missing_field:drone_id
    ├── Has timestamp?         ── NO  → error: missing_field:timestamp
    ├── Has payloads?          ── NO  → error: missing_field:payloads
    │
    ├── drone_id non-empty?    ── NO  → error: invalid_drone_id
    ├── timestamp ISO8601?     ── NO  → error: invalid_timestamp
    ├── payloads non-empty?    ── NO  → error: invalid_payloads
    ├── payloads count OK?     ── NO  → error: too_many_payloads
    ├── signature present?     ── NO (if required) → error: missing_signature
    │
    └── Per payload:
        ├── Has type/filename/mime/size? ─ NO → error
        ├── size_bytes valid?            ─ NO → error
        ├── size_bytes < max?            ─ NO → error
        ├── filename has / or \?         ─ YES → error: path_traversal
        └── MIME in allowed set?         ─ NO → warning
```

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Validation — 4 test cases
# ─────────────────────────────────────────────────────────────────────────────

config = InterceptorConfig()

def show_validation(label, drone_json):
    errors, warnings = validate_submission(drone_json, config)
    status = '\u2705 PASS' if not errors else '\u274c FAIL'
    print(f'\n{status} | {label}')
    if errors:
        for e in errors:
            print(f'     \u274c error: {e}')
    if warnings:
        for w in warnings:
            print(f'     \u26a0\ufe0f  warning: {w}')
    if not errors and not warnings:
        print(f'     No errors, no warnings')

# Test 1: Valid submission
show_validation('Valid submission', {
    'drone_id': 'DRN-001',
    'timestamp': '2025-10-13T03:00:12Z',
    'payloads': [
        {'type': 'video', 'filename': 'flight.mp4', 'mime': 'video/mp4', 'size_bytes': 100000}
    ]
})

# Test 2: Missing required fields
show_validation('Missing fields (no drone_id, no timestamp)', {
    'payloads': [{'type': 'video', 'filename': 'x.mp4', 'mime': 'video/mp4', 'size_bytes': 100}]
})

# Test 3: Path traversal in filename
show_validation('Path traversal attack', {
    'drone_id': 'DRN-001',
    'timestamp': '2025-10-13T03:00:12Z',
    'payloads': [
        {'type': 'text', 'filename': '../../etc/passwd', 'mime': 'text/plain', 'size_bytes': 500}
    ]
})

# Test 4: Oversized payload
show_validation('Oversized payload (600 MB > 500 MB limit)', {
    'drone_id': 'DRN-001',
    'timestamp': '2025-10-13T03:00:12Z',
    'payloads': [
        {'type': 'video', 'filename': 'huge.mp4', 'mime': 'video/mp4',
         'size_bytes': 600_000_000}
    ]
})


✅ PASS | Valid submission
     No errors, no warnings

❌ FAIL | Missing fields (no drone_id, no timestamp)
     ❌ error: missing_field:drone_id
     ❌ error: missing_field:timestamp

❌ FAIL | Path traversal attack
     ❌ error: payload_0:path_traversal_in_filename

❌ FAIL | Oversized payload (600 MB > 500 MB limit)
     ❌ error: payload_0:exceeds_max_size:500000000


---
## Section 6 — Stage 2: Authentication

### 🔵 Theory

After validation passes, we must verify **who sent this data**. A perfectly structured
submission could still come from a compromised or spoofed drone.

Authentication has three components:

1. **Device Registry** — An in-memory (or database-backed) lookup of known drones.
   Each entry contains: `trusted` (bool), `reputation` (0.0–1.0), `revoked` (bool).

2. **Signature Verification** — If the drone provides a cryptographic signature,
   we verify it using HMAC-SHA256 (or Ed25519 in production). This proves the
   data was not tampered with in transit.

3. **Unknown Device Policy** — What to do when a drone is not in the registry:
   - `flag` — Allow through but mark as "unknown" (default)
   - `reject` — Block the submission entirely
   - `allow` — Allow through silently (testing only)

---

### 📊 Authentication Flow

```
drone_id
  │
  ├── Lookup in Device Registry
  │     │
  │     ├── NOT FOUND
  │     │     ├── policy = "reject"  → AuthResult(status="rejected")
  │     │     └── policy = "flag"    → AuthResult(status="unknown")
  │     │
  │     └── FOUND
  │           │
  │           ├── revoked = True     → AuthResult(status="rejected")
  │           │
  │           ├── trusted = True
  │           │     ├── signature valid?  → AuthResult(status="authenticated")
  │           │     └── signature invalid  → downgrade to "untrusted"
  │           │
  │           └── trusted = False    → AuthResult(status="untrusted")
  │
  └── AuthResult includes: status, drone_id, reputation, trusted, details
```

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Authentication — 4 scenarios
# ─────────────────────────────────────────────────────────────────────────────

# Set up device registry
registry = DeviceRegistry(registry={
    'DRN-001': {'trusted': True,  'reputation': 0.9},
    'DRN-002': {'trusted': False, 'reputation': 0.4},
    'DRN-003': {'trusted': True,  'reputation': 0.95, 'revoked': True},
})

def show_auth(label, authenticator, drone_id, signature=None):
    result = authenticator.authenticate(drone_id, signature=signature, payload_hash='abc123')
    icon = {'authenticated': '\u2705', 'untrusted': '\u26a0\ufe0f', 'unknown': '\u2753', 'rejected': '\u274c'}
    print(f"\n{icon.get(result.status, '?')} {label}")
    print(f"   status     : {result.status}")
    print(f"   drone_id   : {result.drone_id}")
    print(f"   trusted    : {result.trusted}")
    print(f"   reputation : {result.reputation}")
    if result.details:
        print(f"   details    : {result.details}")

# Scenario 1: Trusted device
auth_flag = Authenticator(device_registry=registry, unknown_device_policy='flag')
show_auth('Trusted device (DRN-001)', auth_flag, 'DRN-001')

# Scenario 2: Untrusted device
show_auth('Untrusted device (DRN-002)', auth_flag, 'DRN-002')

# Scenario 3: Unknown device with policy=flag
show_auth('Unknown device (DRN-999, policy=flag)', auth_flag, 'DRN-999')

# Scenario 4: Unknown device with policy=reject
auth_reject = Authenticator(device_registry=registry, unknown_device_policy='reject')
show_auth('Unknown device (DRN-999, policy=reject)', auth_reject, 'DRN-999')

# Scenario 5: Revoked device
show_auth('Revoked device (DRN-003)', auth_flag, 'DRN-003')


✅ Trusted device (DRN-001)
   status     : authenticated
   drone_id   : DRN-001
   trusted    : True
   reputation : 0.9
   details    : {'registry_trusted': True, 'signature_check': None, 'firmware_whitelist': None}

⚠️ Untrusted device (DRN-002)
   status     : untrusted
   drone_id   : DRN-002
   trusted    : False
   reputation : 0.4
   details    : {'registry_trusted': False, 'signature_check': None, 'firmware_whitelist': None}

❓ Unknown device (DRN-999, policy=flag)
   status     : unknown
   drone_id   : DRN-999
   trusted    : False
   reputation : None
   details    : {'reason': 'device_not_in_registry', 'policy_applied': 'flag'}

❌ Unknown device (DRN-999, policy=reject)
   status     : rejected
   drone_id   : DRN-999
   trusted    : False
   reputation : None
   details    : {'reason': 'device_not_registered'}

❌ Revoked device (DRN-003)
   status     : rejected
   drone_id   : DRN-003
   trusted    : False
   reputation : 0.95
   details    : {'reason': 'device_revoked'

---
## Section 7 — Stage 3: Metadata Extraction

### 🔵 Theory

Once a submission is validated and authenticated, we extract and normalize its metadata.
This produces structured, clean data that downstream modules can rely on.

There are **four extraction functions**:

| Function | What it extracts | Validation applied |
|----------|-----------------|--------------------|
| `extract_mission_context()` | mission_id, zone, operator, firmware, sensitivity | Normalizes sensitivity level |
| `extract_geo_metadata()` | lat, lon, alt | Rejects invalid coordinates (lat > 90, etc.) |
| `extract_telemetry_summary()` | speed, heading, battery, signal | Flags anomalies (negative speed, battery > 100) |
| `extract_additional_metadata()` | Any extra fields | Strips dangerous keys (`__proto__`, `eval`), truncates long values |

**Telemetry anomaly detection** is particularly important. A compromised drone might
report impossible values (battery = 120%, speed = -5 m/s, heading = 400°). These
anomalies are flagged and passed to the Game-Theoretic Estimator as additional signals.

---

### 📊 Metadata Extraction Flow

```
DroneSubmission
    │
    ├── extract_mission_context()
    │     └── {mission_id, mission_zone, operator_id, firmware_version, sensitivity}
    │
    ├── extract_geo_metadata()
    │     └── {lat, lon, alt} or None (if invalid coordinates)
    │
    ├── extract_telemetry_summary()
    │     └── {speed, heading, battery, signal, telemetry_anomalies: [...]}
    │
    └── extract_additional_metadata()
          └── {key: value, ...} with dangerous keys removed
```

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Metadata Extraction — Normal submission
# ─────────────────────────────────────────────────────────────────────────────

# Build a submission with all metadata fields populated
full_submission = DroneSubmission.from_dict({
    'drone_id': 'DRN-001',
    'timestamp': '2025-10-13T03:00:12Z',
    'mission_id': 'MSN-142',
    'mission_zone': 'zone-a',
    'geo': {'lat': 12.971598, 'lon': 77.594566, 'alt': 120},
    'payloads': [{'type': 'video', 'filename': 'f.mp4', 'mime': 'video/mp4', 'size_bytes': 1000}],
    'telemetry': {'speed': 12.5, 'heading': 145.2, 'battery': 78.4, 'signal_strength': 82.1},
    'firmware_version': 'v1.2.0',
    'operator_id': 'OP-12',
    'additional_metadata': {'camera_model': 'CAM-X1000', 'frame_rate': 30, 'mission_sensitivity': 'HIGH'},
})

print('\u2500' * 65)
print('  METADATA EXTRACTION — Normal Submission')
print('\u2500' * 65)

# 1. Mission context
mission = extract_mission_context(full_submission)
print('\n\u2705 Mission Context:')
for k, v in mission.items():
    print(f'   {k:25s}: {v}')

# 2. Geo metadata
geo_data = extract_geo_metadata(full_submission)
print(f'\n\u2705 Geo Metadata: {geo_data}')

# 3. Telemetry summary
telem = extract_telemetry_summary(full_submission)
print(f'\n\u2705 Telemetry Summary:')
for k, v in telem.items():
    print(f'   {k:25s}: {v}')

# 4. Additional metadata
add_meta = extract_additional_metadata(full_submission)
print(f'\n\u2705 Additional Metadata: {add_meta}')

# --- Now test with anomalous telemetry ---
print('\n' + '\u2500' * 65)
print('  METADATA EXTRACTION — Anomalous Telemetry')
print('\u2500' * 65)

anomalous_submission = DroneSubmission.from_dict({
    'drone_id': 'DRN-999',
    'timestamp': '2025-10-13T04:00:00Z',
    'payloads': [{'type': 'text', 'filename': 'data.json', 'mime': 'application/json', 'size_bytes': 500}],
    'telemetry': {'speed': -5.0, 'heading': 400, 'battery': 120, 'signal_strength': -10},
})

telem_anomalous = extract_telemetry_summary(anomalous_submission)
print(f'\n\u26a0\ufe0f  Telemetry Summary (anomalous):')
for k, v in telem_anomalous.items():
    flag = ' \u274c ANOMALY' if k == 'telemetry_anomalies' else ''
    print(f'   {k:25s}: {v}{flag}')

─────────────────────────────────────────────────────────────────
  METADATA EXTRACTION — Normal Submission
─────────────────────────────────────────────────────────────────

✅ Mission Context:
   mission_id               : MSN-142
   mission_zone             : zone-a
   operator_id              : OP-12
   firmware_version         : v1.2.0
   mission_sensitivity      : high

✅ Geo Metadata: {'lat': 12.971598, 'lon': 77.594566, 'alt': 120.0}

✅ Telemetry Summary:
   speed                    : 12.5
   heading                  : 145.2
   battery                  : 78.4
   signal_strength          : 82.1

✅ Additional Metadata: {'camera_model': 'CAM-X1000', 'frame_rate': 30, 'mission_sensitivity': 'HIGH'}

─────────────────────────────────────────────────────────────────
  METADATA EXTRACTION — Anomalous Telemetry
─────────────────────────────────────────────────────────────────

⚠️  Telemetry Summary (anomalous):
   speed                    : -5.0
   heading                  : 400
   batt

---
## Section 8 — Stage 4: Payload Analysis

### 🔵 Theory

This is the core security analysis stage. Each payload (file) in the submission is
examined against a **9-point security heuristic**. The heuristic is deliberately
lightweight — it operates on metadata alone (filename, MIME type, size, encryption flag)
without opening the actual file. Deep analysis happens later in the Sandbox (Module 4).

**Why 9 checks instead of 1 or 2?**

No single check catches everything. An attacker can bypass extension checking by using
a correct extension. They can bypass MIME checking by setting the right MIME type.
But it is very hard to bypass all 9 checks simultaneously. The combination creates
defense in depth.

---

### 📊 The 9-Point Security Heuristic

| # | Check | Severity | What triggers it | Why it matters |
|---|-------|----------|-----------------|----------------|
| 1 | Encrypted payload | Medium | `encryption=True` | Cannot inspect contents; may hide malware |
| 2 | Nested archive | Medium | `container=True` | ZIP-in-ZIP used for evasion |
| 3 | Large binary | Low | `size_bytes >= 10MB` | Large binaries need resource-aware scanning |
| 4 | Suspicious MIME | High | MIME in `{x-msdownload, x-executable, ...}` | Executable MIME types on a drone feed |
| 5 | Executable file | Critical | Extension in `{exe, dll, bat, ps1, ...}` | Executables should never come from drones |
| 6 | Double extension | High | `photo.jpg.exe` (2+ dots, outer = suspicious) | Classic social engineering technique |
| 7 | Hidden file | Medium | Filename starts with `.` | Unix hidden files may be data exfiltration |
| 8 | Zero-size file | Medium | `size_bytes=0` for video/image/archive | Empty media files are suspicious |
| 9 | MIME-extension mismatch | Medium | MIME says JPEG but extension is `.png` | Possible disguised file |

---

### 📊 Risk Score Computation

```
Severity weights:  critical=1.0  high=0.7  medium=0.4  low=0.2

risk_score = min(1.0, sum(weights) / 3.0)

Examples:
  No flags             → 0.0
  [large_binary]       → 0.2 / 3.0 = 0.067
  [encrypted_payload]  → 0.4 / 3.0 = 0.133
  [executable_file]    → 1.0 / 3.0 = 0.333
  [executable + suspicious_mime + double_extension]
                       → (1.0 + 0.7 + 0.7) / 3.0 = 0.800
```

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Payload Analysis — 5 test payloads
# ─────────────────────────────────────────────────────────────────────────────

config = InterceptorConfig()

test_payloads = [
    ('Clean video',     PayloadEntry(type='video', filename='flight.mp4', mime='video/mp4',
                                     size_bytes=4_500_000, encryption=False, container=False)),
    ('Encrypted archive', PayloadEntry(type='archive', filename='bundle.zip', mime='application/zip',
                                       size_bytes=4_200_000, encryption=True, container=True)),
    ('Executable file',  PayloadEntry(type='unknown', filename='update.exe', mime='application/x-msdownload',
                                      size_bytes=2_500_000, encryption=False, container=False)),
    ('Double extension', PayloadEntry(type='image', filename='photo.jpg.exe', mime='application/x-msdownload',
                                      size_bytes=1_500_000, encryption=False, container=False)),
    ('MIME mismatch',    PayloadEntry(type='image', filename='photo.png', mime='image/jpeg',
                                      size_bytes=320_000, encryption=False, container=False)),
]

print('\u2500' * 78)
print(f'{"Payload":22s} {"Flags":45s} {"Risk":>8s}')
print('\u2500' * 78)

for label, payload in test_payloads:
    flags = analyze_payload(payload, config)
    risk = compute_payload_risk_score(flags)
    flag_str = ', '.join(flags) if flags else '(none)'
    severity_str = ', '.join(f'{f}={FLAG_SEVERITY.get(f, "?")}' for f in flags) if flags else ''
    icon = '\u2705' if not flags else ('\u274c' if risk > 0.5 else '\u26a0\ufe0f')
    print(f'{icon} {label:20s} {flag_str:45s} {risk:7.3f}')
    if severity_str:
        print(f'   severities: {severity_str}')

print('\u2500' * 78)

──────────────────────────────────────────────────────────────────────────────
Payload                Flags                                             Risk
──────────────────────────────────────────────────────────────────────────────
✅ Clean video          (none)                                          0.000
⚠️ Encrypted archive    encrypted_payload, nested_archive               0.267
   severities: encrypted_payload=medium, nested_archive=medium
❌ Executable file      suspicious_mime, executable_file                0.567
   severities: suspicious_mime=high, executable_file=critical
❌ Double extension     suspicious_mime, executable_file, double_extension   0.800
   severities: suspicious_mime=high, executable_file=critical, double_extension=high
⚠️ MIME mismatch        mime_extension_mismatch                         0.133
   severities: mime_extension_mismatch=medium
──────────────────────────────────────────────────────────────────────────────


In [14]:
# ─── DEMO: Threat notes generation ────────────────────────────────────────────
# generate_threat_notes() combines flags + auth status into human-readable notes

print('Threat Notes Examples:')
print(f'  No flags, authenticated  : "{generate_threat_notes([], "authenticated")}')
print(f'  No flags, unknown device : "{generate_threat_notes([], "unknown")}"')
print(f'  Encrypted + untrusted    : "{generate_threat_notes(["encrypted_payload"], "untrusted")}"')
print(f'  Executable + suspicious  : "{generate_threat_notes(["executable_file", "suspicious_mime", "double_extension"], "unknown")}"')

Threat Notes Examples:
  No flags, authenticated  : "normal feed
  No flags, unknown device : "device unknown"
  Encrypted + untrusted    : "device untrusted; defer analysis: encrypted or nested contents"
  Executable + suspicious  : "device unknown; CRITICAL: executable_file; high-risk: suspicious_mime, double_extension"


---
## Section 9 — Stage 5: Checksum Verification

### 🔵 Theory

Checksums detect **tampered files**. If an attacker intercepts a drone's data stream
and modifies a file in transit (man-in-the-middle attack), the checksum will not match.

**How it works:**
1. The drone computes a hash (SHA-256) of each file before sending
2. The hash is included in the submission JSON (`checksum` field)
3. When we receive the file, we recompute the hash and compare

```
Drone side:                         Server side:
  file bytes ─► SHA-256 ─► hash_A      file bytes ─► SHA-256 ─► hash_B
                                      
  If hash_A == hash_B  → \u2705 integrity confirmed
  If hash_A != hash_B  → \u274c possible tampering
  If hash_A is absent  → \u2753 verification skipped
```

The checksum verifier supports three algorithms:
- **SHA-256** (default, recommended)
- **SHA-1** (legacy, weaker)
- **MD5** (legacy, collision-prone — use only for backward compatibility)

**Stub checksums** (e.g., `a1b2c3d4...`) are automatically skipped — they indicate
the drone did not compute a real hash.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Checksum Verification
# ─────────────────────────────────────────────────────────────────────────────

demo_dir = tempfile.mkdtemp(prefix='checksum_demo_')

# Create a test file
test_file = os.path.join(demo_dir, 'telemetry.json')
test_content = b'{"speed": 12.5, "heading": 145.2, "battery": 78.4}'
with open(test_file, 'wb') as f:
    f.write(test_content)

# 1. Compute checksum from bytes
checksum_bytes = compute_bytes_checksum(test_content, 'sha256')
print(f'\u2705 SHA-256 from bytes: {checksum_bytes}')

# 2. Compute checksum from file
checksum_file = compute_file_checksum(test_file, 'sha256')
print(f'\u2705 SHA-256 from file : {checksum_file}')
print(f'   Match: {checksum_bytes == checksum_file}')

# 3. Verify correct checksum
result = verify_checksum(test_file, checksum_bytes, 'sha256')
print(f'\n\u2705 Verify correct checksum : {result}')

# 4. Verify wrong checksum (simulating tampering)
result_bad = verify_checksum(test_file, 'deadbeefdeadbeefdeadbeef', 'sha256')
print(f'\u274c Verify tampered checksum: {result_bad}')

# 5. Verify stub checksum (should be skipped)
result_stub = verify_checksum(test_file, 'a1b2c3d4...', 'sha256')
print(f'\u2753 Verify stub checksum    : {result_stub} (None = skipped)')

# 6. Different algorithms
print(f'\n  SHA-1  : {compute_bytes_checksum(test_content, "sha1")}')
print(f'  MD5    : {compute_bytes_checksum(test_content, "md5")}')
print(f'  SHA-512: {compute_bytes_checksum(test_content, "sha512")[:64]}...')

# Cleanup
import shutil
shutil.rmtree(demo_dir)

Checksum mismatch for /tmp/checksum_demo_749p3xzl/telemetry.json: expected=deadbeefdeadbeefdeadbeef, actual=061a024e3fab5180ed0d5ed4aa3cc6254d4f92476f67180c6c148b1d8b9fc2c5


✅ SHA-256 from bytes: 061a024e3fab5180ed0d5ed4aa3cc6254d4f92476f67180c6c148b1d8b9fc2c5
✅ SHA-256 from file : 061a024e3fab5180ed0d5ed4aa3cc6254d4f92476f67180c6c148b1d8b9fc2c5
   Match: True

✅ Verify correct checksum : True
❌ Verify tampered checksum: False
❓ Verify stub checksum    : None (None = skipped)

  SHA-1  : bf37f1f7641a2e7468c010c817ec2a1d2fae4654
  MD5    : 0416f4a705513eb7b6932bbaceaf4e67
  SHA-512: 7f22fb3fa57d8a2366cfa446900ca691572bd30e9427b0359170d63207c5a5b3...


---
## Section 10 — Stage 6: Artifact Management

### 🔵 Theory

After analysis, each payload is cataloged as an **artifact** with a unique ID,
storage pointer, and thumbnail reference (for visual payloads).

This is not just record-keeping — it enables:
- **Forensic traceability** — every file can be traced back to its source drone and mission
- **Downstream referencing** — the Sandbox, AI/ML classifier, and Response Manager
  all reference artifacts by their `artifact_id`
- **Storage abstraction** — code works the same whether storage is local filesystem,
  S3, or MinIO

```
PayloadEntry  ──►  create_artifact_record()  ──►  ArtifactRecord

  Fields added:
  ├── artifact_id      : "artifact://a1b2c3d4e5f6"  (globally unique)
  ├── security_flags   : ["encrypted_payload", ...]  (from Stage 4)
  ├── checksum_verified: True/False/None             (from Stage 5)
  ├── pointer_storage  : "s3://forensics/artifacts/DRN-001/..."  (where the file lives)
  └── thumbnail        : "thumb://abc123"            (for video/image payloads)
```

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Artifact Management
# ─────────────────────────────────────────────────────────────────────────────

s3_config = InterceptorConfig(storage_backend='s3')
fs_config = InterceptorConfig(storage_backend='filesystem')

# Sample payloads
video_payload = PayloadEntry(
    type='video', filename='flight_01.mp4', mime='video/mp4',
    size_bytes=4_500_000, encryption=False, container=False,
)
text_payload = PayloadEntry(
    type='text', filename='notes.txt', mime='text/plain',
    size_bytes=2048, encryption=False, container=False,
)

# Generate artifact IDs
print('Artifact ID examples:')
for _ in range(3):
    print(f'  {generate_artifact_id()}')

# Ingest IDs
print(f'\nIngest ID example: {generate_ingest_id()}')

# Thumbnail generation
print(f'\nThumbnails:')
print(f'  Video payload : {generate_thumbnail_ref(video_payload)}')
print(f'  Text payload  : {generate_thumbnail_ref(text_payload)} (None = no thumbnail for text)')

# Storage pointers
print(f'\nStorage pointers:')
print(f'  S3 backend    : {build_storage_pointer(video_payload, "DRN-001", "ingest_abc", s3_config)}')
print(f'  Filesystem    : {build_storage_pointer(video_payload, "DRN-001", "ingest_abc", fs_config)}')

# Full artifact record
print('\n' + '\u2500' * 65)
print('  Full ArtifactRecord')
print('\u2500' * 65)

artifact = create_artifact_record(
    payload=video_payload,
    security_flags=['large_binary'],
    drone_id='DRN-001',
    ingest_id='ingest_abc123',
    config=s3_config,
    checksum_verified=True,
)

for k, v in artifact.to_dict().items():
    print(f'  {k:20s}: {v}')

Artifact ID examples:
  artifact://b5c206e5e4914d01
  artifact://cf0238c1db384f75
  artifact://a7bc74c5e6634eb4

Ingest ID example: ingest_018712b7fe5b

Thumbnails:
  Video payload : thumb://298b0492febe
  Text payload  : None (None = no thumbnail for text)

Storage pointers:
  S3 backend    : s3://forensics/artifacts/DRN-001/ingest_abc/98af13e1_flight_01.mp4
  Filesystem    : drone_remote_store/DRN-001/flight_01.mp4

─────────────────────────────────────────────────────────────────
  Full ArtifactRecord
─────────────────────────────────────────────────────────────────
  artifact_id         : artifact://0a7a94dc395e4f9d
  filename            : flight_01.mp4
  type                : video
  mime                : video/mp4
  size_bytes          : 4500000
  encryption          : False
  container           : False
  security_flags      : ['large_binary']
  checksum_verified   : True
  thumbnail           : thumb://c02e331e2bfc
  pointer_storage     : s3://forensics/artifacts/DRN-001//4508b

---
## Section 11 — Uplink Communication

### 🔵 Theory

The BEL proposal requires **two-way communication** between the ingestion point
and the central control center. The "uplink" channel allows the control center
to send commands that modify interceptor behavior in real time.

This is critical for **adaptive security**:
- A threat intelligence feed detects a new attack pattern → control center sends
  `QUARANTINE` for affected ingestions
- A drone is compromised → control center sends `REVOKE_DEVICE` to block it
- A mission zone becomes hostile → control center sends `UPDATE_ZONE_RISK`

---

### 📊 Uplink Architecture

```
┌───────────────────────┐          ┌────────────────────────────┐
│  Control Center       │          │  Ingestion Interceptor       │
│                       │          │                              │
│  Sends commands:      │ ────►   │  UplinkReceiver               │
│  - QUARANTINE         │          │    │                           │
│  - RELEASE            │          │    ▼                           │
│  - REVOKE_DEVICE      │          │  UplinkCommandHandler         │
│  - UPDATE_ZONE_RISK   │          │    ├── quarantine ingest       │
│  - FORCE_RESCAN       │          │    ├── revoke device           │
│  - UPDATE_CONFIG      │          │    ├── update zone risk        │
│                       │          │    └── release ingest          │
└───────────────────────┘          └────────────────────────────┘
```

**Command types:**

| Command | Target | Parameters | Effect |
|---------|--------|-----------|--------|
| `QUARANTINE` | ingest_id | — | Marks ingestion as quarantined |
| `RELEASE` | ingest_id | — | Removes quarantine flag |
| `REVOKE_DEVICE` | drone_id | — | Sets device as revoked in registry |
| `UPDATE_ZONE_RISK` | `*` | `{zone, risk}` | Updates zone risk score |
| `FORCE_RESCAN` | ingest_id | — | Queues for re-analysis |
| `UPDATE_CONFIG` | `*` | `{key: value}` | Updates config parameters |

⚠️ **Note:** In the current implementation, the uplink uses an in-memory queue.
In production, this would be replaced by gRPC, MQTT, or a REST polling mechanism.

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Uplink Communication
# ─────────────────────────────────────────────────────────────────────────────

# Set up receiver and handler
receiver = UplinkReceiver(mode='memory')

# The handler needs an authenticator so it can revoke devices
demo_registry = DeviceRegistry(registry={
    'DRN-001': {'trusted': True, 'reputation': 0.9},
    'DRN-002': {'trusted': True, 'reputation': 0.7},
})
demo_authenticator = Authenticator(device_registry=demo_registry)

handler = UplinkCommandHandler(
    authenticator=demo_authenticator,
    zone_risk_lookup={'zone-a': 0.6, 'zone-b': 0.2},
)

# --- Command 1: Quarantine an ingestion ---
cmd1 = UplinkCommand(
    command_type=CommandType.QUARANTINE,
    target='ingest_abc123',
    command_id='cmd-001',
)
receiver.push_command(cmd1)
result1 = handler.handle(cmd1)
print(f'\u2705 QUARANTINE: {result1}')
print(f'   Quarantined set: {handler.quarantined_ingests}')

# --- Command 2: Update zone risk ---
cmd2 = UplinkCommand(
    command_type=CommandType.UPDATE_ZONE_RISK,
    target='*',
    parameters={'zone': 'zone-a', 'risk': 0.95},
    command_id='cmd-002',
)
receiver.push_command(cmd2)
result2 = handler.handle(cmd2)
print(f'\n\u2705 UPDATE_ZONE_RISK: {result2}')
print(f'   Zone risks now: {handler.zone_risk_lookup}')

# --- Command 3: Revoke a device ---
cmd3 = UplinkCommand(
    command_type=CommandType.REVOKE_DEVICE,
    target='DRN-002',
    command_id='cmd-003',
)
receiver.push_command(cmd3)
result3 = handler.handle(cmd3)
print(f'\n\u2705 REVOKE_DEVICE: {result3}')
# Verify: try to authenticate the revoked device
auth_after = demo_authenticator.authenticate('DRN-002')
print(f'   DRN-002 auth after revoke: status={auth_after.status}')

# --- Command 4: Release quarantine ---
cmd4 = UplinkCommand(
    command_type=CommandType.RELEASE,
    target='ingest_abc123',
    command_id='cmd-004',
)
result4 = handler.handle(cmd4)
print(f'\n\u2705 RELEASE: {result4}')
print(f'   Quarantined set after release: {handler.quarantined_ingests}')

# --- Poll commands ---
print(f'\n\u2500' * 40)
pending = receiver.poll_commands()
print(f'Pending commands in queue: {len(pending)}')
for cmd in pending:
    receiver.acknowledge(cmd.command_id)
print(f'After acknowledging all: {len(receiver.poll_commands())} pending')

✅ QUARANTINE: {'status': 'quarantined', 'target': 'ingest_abc123'}
   Quarantined set: {'ingest_abc123'}

✅ UPDATE_ZONE_RISK: {'status': 'updated', 'zone': 'zone-a', 'risk': 0.95}
   Zone risks now: {'zone-a': 0.95, 'zone-b': 0.2}

✅ REVOKE_DEVICE: {'status': 'revoked', 'target': 'DRN-002'}
   DRN-002 auth after revoke: status=rejected

✅ RELEASE: {'status': 'released', 'target': 'ingest_abc123'}
   Quarantined set after release: set()

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
Pending commands in queue: 3
After acknowledging all: 0 pending


---
## Section 12 — Full Pipeline: `IngestionInterceptor` Class

### 🔵 Theory

The `IngestionInterceptor` class is the **orchestrator** that ties all 6 stages together.
It is what the rest of the system calls when a drone submission arrives.

**Interface:**
```python
interceptor = IngestionInterceptor(
    config=InterceptorConfig(...),
    device_registry={'DRN-001': {'trusted': True, 'reputation': 0.9}},
    zone_risk_lookup={'zone-a': 0.6},
)
result = interceptor.process(drone_json)
# result.success           → True/False
# result.ingest_metadata   → IngestMetadata with all extracted info
# result.artifact_records  → List[ArtifactRecord] with per-file analysis
# result.errors / warnings → Any issues found
```

The class also:
- Processes pending **uplink commands** before each submission
- Maintains **statistics** (total processed, rejected, flagged)
- Exposes the **authenticator** and **uplink receiver** for direct access
- Supports **batch processing** via `process_batch()`

---

### 📊 Orchestration Sequence

```
interceptor.process(drone_json)
    │
    ├── Process pending uplink commands
    │
    ├── Stage 1: validate_submission(drone_json, config)
    │     └── errors? → return IngestResult(success=False)
    │
    ├── Parse into DroneSubmission.from_dict()
    │
    ├── Stage 2: authenticator.authenticate(drone_id, signature)
    │     └── rejected? → return IngestResult(success=False)
    │
    ├── Stage 3: extract metadata (mission, geo, telemetry, additional)
    │
    ├── For each payload:
    │     ├── Stage 4: analyze_payload() → security flags
    │     ├── Stage 5: verify_checksum() → True/False/None
    │     └── Stage 6: create_artifact_record()
    │
    ├── Build IngestMetadata with all results
    │
    └── return IngestResult(success=True, metadata, artifacts, warnings)
```

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: Full Pipeline — single submission end-to-end
# ─────────────────────────────────────────────────────────────────────────────

interceptor = IngestionInterceptor(
    config=InterceptorConfig(
        require_signature=False,
        verify_checksums=True,
        storage_backend='s3',
    ),
    device_registry={
        'DRN-001': {'trusted': True, 'reputation': 0.9},
    },
    zone_risk_lookup={'zone-a': 0.6},
)

drone_json = {
    'drone_id': 'DRN-001',
    'timestamp': '2025-10-13T03:00:12Z',
    'mission_id': 'MSN-142',
    'mission_zone': 'zone-a',
    'geo': {'lat': 12.971598, 'lon': 77.594566, 'alt': 120},
    'payloads': [
        {'type': 'video', 'filename': 'drn001_fpv_001.mp4', 'mime': 'video/mp4',
         'size_bytes': 4500000, 'encryption': False, 'container': False,
         'checksum': 'a1b2c3d4...'},
        {'type': 'image', 'filename': 'drn001_cam_001.jpg', 'mime': 'image/jpeg',
         'size_bytes': 320000, 'encryption': False, 'container': False,
         'checksum': 'e5f6g7h8...'},
    ],
    'telemetry': {'speed': 12.5, 'heading': 145.2, 'battery': 78.4, 'signal_strength': 82.1},
    'firmware_version': 'v1.2.0',
    'operator_id': 'OP-12',
    'additional_metadata': {'camera_model': 'CAM-X1000', 'frame_rate': 30},
}

result = interceptor.process(drone_json)

print('=' * 70)
print('  FULL PIPELINE RESULT')
print('=' * 70)
print(f'  Success         : {result.success}')
print(f'  Warnings        : {result.warnings}')
print(f'  Errors          : {result.errors}')

if result.success:
    meta = result.ingest_metadata
    print(f'\n  Ingest ID       : {meta.ingest_id}')
    print(f'  Drone ID        : {meta.drone_id}')
    print(f'  Timestamp       : {meta.timestamp}')
    print(f'  Received at     : {meta.received_at}')
    print(f'  Mission ID      : {meta.mission_id}')
    print(f'  Mission Zone    : {meta.mission_zone}')
    print(f'  Geo             : {meta.geo}')
    print(f'  Operator        : {meta.operator_id}')
    print(f'  Firmware        : {meta.firmware_version}')
    print(f'  Files           : {meta.num_files} ({meta.total_size_bytes:,} bytes)')
    print(f'  Auth Result     : {meta.auth_result}')
    print(f'  Reputation      : {meta.reputation}')
    print(f'  Zone Risk       : {meta.zone_risk}')
    print(f'  Security Flags  : {meta.insecure_flags}')
    print(f'  Notes           : {meta.notes}')

    print(f'\n  Artifacts ({len(result.artifact_records)}):')
    for i, art in enumerate(result.artifact_records, 1):
        print(f'    [{i}] {art.filename} ({art.type}, {art.size_bytes:,} bytes)')
        print(f'        artifact_id  : {art.artifact_id}')
        print(f'        security_flags: {art.security_flags}')
        print(f'        checksum_ok  : {art.checksum_verified}')
        print(f'        storage      : {art.pointer_storage}')
        print(f'        thumbnail    : {art.thumbnail}')

print(f'\n  Pipeline Stats  : {interceptor.stats}')
print('=' * 70)

2026-04-02 00:45:10,871 [ingestion_interceptor.interceptor] INFO: Processed ingest_fd58ea04cc00 from DRN-001 in 0.001s | flags=[] | auth=authenticated


  FULL PIPELINE RESULT
  Success         : True
  Warnings        : []
  Errors          : []

  Ingest ID       : ingest_fd58ea04cc00
  Drone ID        : DRN-001
  Timestamp       : 2025-10-13T03:00:12Z
  Received at     : 2026-04-01T19:15:10.870467+00:00
  Mission ID      : MSN-142
  Mission Zone    : zone-a
  Geo             : {'lat': 12.971598, 'lon': 77.594566, 'alt': 120.0}
  Operator        : OP-12
  Firmware        : v1.2.0
  Files           : 2 (4,820,000 bytes)
  Auth Result     : authenticated
  Reputation      : 0.9
  Zone Risk       : 0.6
  Security Flags  : []
  Notes           : normal feed

  Artifacts (2):
    [1] drn001_fpv_001.mp4 (video, 4,500,000 bytes)
        artifact_id  : artifact://4da04b49e6474cc3
        security_flags: []
        checksum_ok  : None
        storage      : s3://forensics/artifacts/DRN-001//eba7d145_drn001_fpv_001.mp4
        thumbnail    : thumb://d78c18bb0654
    [2] drn001_cam_001.jpg (image, 320,000 bytes)
        artifact_id  : artifact:

---
## Section 13 — End-to-End Test with Multiple Scenarios

Five realistic drone submissions covering the full range of scenarios:

| Scenario | Drone | Description | Expected outcome |
|----------|-------|-------------|------------------|
| A | DRN-001 | Normal trusted drone, video + image | Clean pass, no flags |
| B | DRN-002 | Encrypted archive from untrusted source | Flags: encrypted, nested, untrusted |
| C | DRN-003 | Telemetry-only routine patrol | Minimal, clean |
| D | DRN-004 | Large video from trusted drone | Flag: large_binary |
| E | DRN-999 | Unknown drone, executable + suspicious MIME | Multiple critical flags |

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# END-TO-END TEST — 5 REALISTIC DRONE SUBMISSIONS
# ─────────────────────────────────────────────────────────────────────────────

DEVICE_REGISTRY = {
    'DRN-001': {'trusted': True,  'reputation': 0.9},
    'DRN-002': {'trusted': False, 'reputation': 0.4},
    'DRN-003': {'trusted': True,  'reputation': 0.95},
    'DRN-004': {'trusted': True,  'reputation': 0.85},
}

ZONE_RISK = {'zone-a': 0.6, 'zone-b': 0.2, 'zone-c': 0.8}

interceptor = IngestionInterceptor(
    config=InterceptorConfig(
        require_signature=False,
        verify_checksums=True,
        storage_backend='s3',
    ),
    device_registry=DEVICE_REGISTRY,
    zone_risk_lookup=ZONE_RISK,
)

SCENARIOS = {
    'A': {
        'label': 'Normal trusted drone — video + image (clean)',
        'json': {
            'drone_id': 'DRN-001',
            'timestamp': '2025-10-13T03:00:12Z',
            'mission_id': 'MSN-142',
            'mission_zone': 'zone-a',
            'geo': {'lat': 12.971598, 'lon': 77.594566, 'alt': 120},
            'payloads': [
                {'type': 'video', 'filename': 'drn001_fpv_001.mp4', 'mime': 'video/mp4',
                 'size_bytes': 4500000, 'encryption': False, 'container': False,
                 'checksum': 'a1b2c3d4...'},
                {'type': 'image', 'filename': 'drn001_cam_001.jpg', 'mime': 'image/jpeg',
                 'size_bytes': 320000, 'encryption': False, 'container': False,
                 'checksum': 'e5f6g7h8...'},
            ],
            'telemetry': {'speed': 12.5, 'heading': 145.2, 'battery': 78.4, 'signal_strength': 82.1},
            'firmware_version': 'v1.2.0',
            'operator_id': 'OP-12',
            'additional_metadata': {'camera_model': 'CAM-X1000', 'frame_rate': 30},
        },
    },
    'B': {
        'label': 'Encrypted archive from untrusted source (suspicious)',
        'json': {
            'drone_id': 'DRN-002',
            'timestamp': '2025-10-13T03:05:45Z',
            'mission_id': 'MSN-143',
            'mission_zone': 'zone-c',
            'geo': {'lat': 13.035542, 'lon': 77.597100, 'alt': 85},
            'payloads': [
                {'type': 'archive', 'filename': 'payload_bundle.zip', 'mime': 'application/zip',
                 'size_bytes': 4200000, 'encryption': True, 'container': True,
                 'checksum': '9f8e7d6c...'},
                {'type': 'text', 'filename': 'notes.txt', 'mime': 'text/plain',
                 'size_bytes': 2048, 'encryption': False, 'container': False,
                 'checksum': '1234abcd...'},
            ],
            'telemetry': {'speed': 0.0, 'heading': 0.0, 'battery': 56.1, 'signal_strength': 65.3},
            'firmware_version': 'v1.1.9',
            'operator_id': 'OP-23',
            'additional_metadata': {'mission_priority': 'high'},
        },
    },
    'C': {
        'label': 'Telemetry-only routine patrol (minimal)',
        'json': {
            'drone_id': 'DRN-003',
            'timestamp': '2025-10-13T03:10:03Z',
            'mission_id': 'MSN-144',
            'mission_zone': 'zone-b',
            'geo': {'lat': 12.967800, 'lon': 77.601200, 'alt': 35},
            'payloads': [
                {'type': 'telemetry', 'filename': 'telemetry_snapshot.json',
                 'mime': 'application/json', 'size_bytes': 1500,
                 'encryption': False, 'container': False, 'checksum': 'fedcba987...'},
            ],
            'telemetry': {'speed': 6.2, 'heading': 220.0, 'battery': 92.3, 'signal_strength': 90.4},
            'firmware_version': 'v1.2.3',
            'operator_id': 'OP-33',
            'additional_metadata': {'note': 'routine patrol', 'weather': 'clear'},
        },
    },
    'D': {
        'label': 'Large video from trusted drone (borderline)',
        'json': {
            'drone_id': 'DRN-004',
            'timestamp': '2025-10-13T03:15:22Z',
            'mission_id': 'MSN-145',
            'mission_zone': 'zone-a',
            'geo': {'lat': 12.975000, 'lon': 77.590000, 'alt': 200},
            'payloads': [
                {'type': 'video', 'filename': 'survey_coverage_long.mp4', 'mime': 'video/mp4',
                 'size_bytes': 12500000, 'encryption': False, 'container': False,
                 'checksum': 'aaaabbbbcccc...'},
                {'type': 'image', 'filename': 'survey_frame_2345.jpg', 'mime': 'image/jpeg',
                 'size_bytes': 550000, 'encryption': False, 'container': False,
                 'checksum': 'ddddeeeeffff...'},
            ],
            'telemetry': {'speed': 8.1, 'heading': 98.7, 'battery': 64.0, 'signal_strength': 75.0},
            'firmware_version': 'v2.0.0',
            'operator_id': 'OP-05',
            'additional_metadata': {'camera_model': 'CAM-PRO-4k', 'mission_sensitivity': 'critical'},
        },
    },
    'E': {
        'label': 'Unknown drone — executable + suspicious MIME (high risk)',
        'json': {
            'drone_id': 'DRN-999',
            'timestamp': '2025-10-13T04:00:00Z',
            'mission_id': 'MSN-666',
            'mission_zone': 'zone-c',
            'geo': {'lat': 12.980000, 'lon': 77.600000, 'alt': 50},
            'payloads': [
                {'type': 'archive', 'filename': 'data.tar.gz.exe',
                 'mime': 'application/x-msdownload',
                 'size_bytes': 15000000, 'encryption': True, 'container': True,
                 'checksum': 'deadbeef...'},
            ],
            'telemetry': {'speed': -5.0, 'heading': 400, 'battery': 120, 'signal_strength': -10},
            'firmware_version': 'v0.0.1',
            'operator_id': 'OP-00',
            'additional_metadata': {'mission_sensitivity': 'critical'},
        },
    },
}

print('=' * 100)
print('  END-TO-END PIPELINE TEST — 5 DRONE SUBMISSIONS')
print('=' * 100)

results_summary = []

for key, scenario in SCENARIOS.items():
    print(f"\n{'\u2500' * 100}")
    print(f"  Scenario {key}: {scenario['label']}")
    print(f"  drone_id={scenario['json']['drone_id']}")
    print(f"{'\u2500' * 100}")

    result = interceptor.process(scenario['json'])

    if not result.success:
        print(f"  \u274c REJECTED: {result.errors}")
        results_summary.append({
            'scenario': key, 'success': False, 'drone': scenario['json']['drone_id'],
            'auth': 'N/A', 'flags': 'N/A', 'notes': str(result.errors),
        })
        continue

    meta = result.ingest_metadata
    print(f"  \u2705 Ingest ID     : {meta.ingest_id}")
    print(f"     Auth Result   : {meta.auth_result}")
    print(f"     Reputation    : {meta.reputation}")
    print(f"     Zone Risk     : {meta.zone_risk}")
    print(f"     Files         : {meta.num_files} ({meta.total_size_bytes:,} bytes)")
    print(f"     Security Flags: {meta.insecure_flags}")
    print(f"     Notes         : {meta.notes}")

    if result.warnings:
        print(f"     Warnings      : {result.warnings}")

    for i, art in enumerate(result.artifact_records, 1):
        print(f"     Artifact [{i}]  : {art.filename} | flags={art.security_flags} | checksum_ok={art.checksum_verified}")

    results_summary.append({
        'scenario': key, 'success': True, 'drone': meta.drone_id,
        'auth': meta.auth_result, 'flags': meta.insecure_flags,
        'notes': meta.notes, 'reputation': meta.reputation,
        'zone_risk': meta.zone_risk,
    })

  END-TO-END PIPELINE TEST — 5 DRONE SUBMISSIONS

────────────────────────────────────────────────────────────────────────────────────────────────────
  Scenario A: Normal trusted drone — video + image (clean)
  drone_id=DRN-001
────────────────────────────────────────────────────────────────────────────────────────────────────


2026-04-02 00:45:10,891 [ingestion_interceptor.interceptor] INFO: Processed ingest_23a25c60e79c from DRN-001 in 0.000s | flags=[] | auth=authenticated
2026-04-02 00:45:10,892 [ingestion_interceptor.interceptor] INFO: Processed ingest_2cc8c711c62c from DRN-002 in 0.000s | flags=['encrypted_payload', 'nested_archive'] | auth=untrusted
2026-04-02 00:45:10,892 [ingestion_interceptor.interceptor] INFO: Processed ingest_2f2869f8e188 from DRN-003 in 0.000s | flags=[] | auth=authenticated
2026-04-02 00:45:10,893 [ingestion_interceptor.interceptor] INFO: Processed ingest_a1f561fbe39e from DRN-004 in 0.000s | flags=['large_binary'] | auth=authenticated
2026-04-02 00:45:10,893 [ingestion_interceptor.authenticator] INFO: Device DRN-999 not found in registry (policy: flag)
2026-04-02 00:45:10,894 [ingestion_interceptor.interceptor] INFO: Processed ingest_4bcdc6d65d7d from DRN-999 in 0.001s | flags=['double_extension', 'encrypted_payload', 'executable_file', 'large_binary', 'nested_archive', 'suspic

  ✅ Ingest ID     : ingest_23a25c60e79c
     Auth Result   : authenticated
     Reputation    : 0.9
     Zone Risk     : 0.6
     Files         : 2 (4,820,000 bytes)
     Security Flags: []
     Notes         : normal feed
     Artifact [1]  : drn001_fpv_001.mp4 | flags=[] | checksum_ok=None
     Artifact [2]  : drn001_cam_001.jpg | flags=[] | checksum_ok=None

────────────────────────────────────────────────────────────────────────────────────────────────────
  Scenario B: Encrypted archive from untrusted source (suspicious)
  drone_id=DRN-002
────────────────────────────────────────────────────────────────────────────────────────────────────
  ✅ Ingest ID     : ingest_2cc8c711c62c
     Auth Result   : untrusted
     Reputation    : 0.4
     Zone Risk     : 0.8
     Files         : 2 (4,202,048 bytes)
     Security Flags: ['encrypted_payload', 'nested_archive']
     Notes         : device untrusted; defer analysis: encrypted or nested contents
     Artifact [1]  : payload_bundle.zip |

In [20]:
# ─── Results Summary Table ──────────────────────────────────────────────────────

print('\n' + '=' * 100)
print('  RESULTS SUMMARY')
print('=' * 100)
print(f'{"Sc":3s} {"Drone":9s} {"Auth":15s} {"Rep":5s} {"Zone":5s} {"Flags":45s} {"Notes"}')
print('\u2500' * 100)

for r in results_summary:
    if not r['success']:
        print(f"{r['scenario']:3s} {r['drone']:9s} {'REJECTED':15s} {'N/A':5s} {'N/A':5s} {'N/A':45s} {r['notes']}")
    else:
        flags_str = ', '.join(r['flags']) if r['flags'] else '(none)'
        rep = f"{r.get('reputation', 'N/A')}"
        zr = f"{r.get('zone_risk', 'N/A')}"
        print(f"{r['scenario']:3s} {r['drone']:9s} {r['auth']:15s} {rep:5s} {zr:5s} {flags_str:45s} {r['notes']}")

print('\u2500' * 100)
print(f'\n  Pipeline Stats: {interceptor.stats}')


  RESULTS SUMMARY
Sc  Drone     Auth            Rep   Zone  Flags                                         Notes
────────────────────────────────────────────────────────────────────────────────────────────────────
A   DRN-001   authenticated   0.9   0.6   (none)                                        normal feed
B   DRN-002   untrusted       0.4   0.8   encrypted_payload, nested_archive             device untrusted; defer analysis: encrypted or nested contents
C   DRN-003   authenticated   0.95  0.2   (none)                                        normal feed
D   DRN-004   authenticated   0.85  0.6   large_binary                                  large binary - consider selective sampling
E   DRN-999   unknown         None  0.8   double_extension, encrypted_payload, executable_file, large_binary, nested_archive, suspicious_mime device unknown; CRITICAL: executable_file; high-risk: double_extension, suspicious_mime; defer analysis: encrypted or nested contents
────────────────────────────

In [21]:
# ─── How output feeds into Game-Theoretic Estimator (Module 2) ─────────────────
#
# ⚠️ MODULE BOUNDARY:
# The Game-Theoretic Threat Estimator is a SEPARATE module.
# The function below is a STUB showing what data the estimator receives
# from the Ingestion Interceptor's output.

def stub_game_theoretic_estimator(ingest_result):
    """
    NOT_IMPLEMENTED — stub only.
    Shows the interface between Module 1 (Ingestion Interceptor)
    and Module 2 (Game-Theoretic Threat Estimator).

    The estimator uses:
    - security_flags: what anomalies were detected
    - auth_result: was the device trusted?
    - reputation: device reputation score
    - zone_risk: how risky is the mission zone
    - telemetry anomalies: impossible sensor values
    - num_files, total_size: payload characteristics
    """
    result_dict = ingest_result.to_dict()
    if 'error' in result_dict:
        return {'threat_level': 'BLOCKED', 'reason': 'rejected at ingestion'}

    meta = result_dict['ingest_metadata']
    return {
        'ingest_id': meta['ingest_id'],
        'inputs_to_estimator': {
            'security_flags': meta['insecure_flags'],
            'auth_result': meta['auth_result'],
            'reputation': meta.get('reputation'),
            'zone_risk': meta.get('zone_risk'),
            'num_files': meta['num_files'],
            'total_size_bytes': meta['total_size_bytes'],
        },
        'threat_level': 'PLACEHOLDER — computed by Game-Theoretic module',
    }

# Demo: pass Scenario E's result to the estimator
scenario_e_result = interceptor.process(SCENARIOS['E']['json'])
estimator_input = stub_game_theoretic_estimator(scenario_e_result)

print('\u2500' * 70)
print('  DATA PASSED TO GAME-THEORETIC THREAT ESTIMATOR')
print('\u2500' * 70)
print(json.dumps(estimator_input, indent=2))
print('\u2500' * 70)
print('\n\u26a0\ufe0f  The threat_level field will be filled by the Game-Theoretic module.')
print('   It uses Bayesian game theory to compute attacker/defender strategies.')

2026-04-02 00:45:10,911 [ingestion_interceptor.authenticator] INFO: Device DRN-999 not found in registry (policy: flag)
2026-04-02 00:45:10,912 [ingestion_interceptor.interceptor] INFO: Processed ingest_2e37aa65171f from DRN-999 in 0.001s | flags=['double_extension', 'encrypted_payload', 'executable_file', 'large_binary', 'nested_archive', 'suspicious_mime'] | auth=unknown


──────────────────────────────────────────────────────────────────────
  DATA PASSED TO GAME-THEORETIC THREAT ESTIMATOR
──────────────────────────────────────────────────────────────────────
{
  "ingest_id": "ingest_2e37aa65171f",
  "inputs_to_estimator": {
    "security_flags": [
      "double_extension",
      "encrypted_payload",
      "executable_file",
      "large_binary",
      "nested_archive",
      "suspicious_mime"
    ],
    "auth_result": "unknown",
    "reputation": null,
    "zone_risk": 0.8,
    "num_files": 1,
    "total_size_bytes": 15000000
  },
  "threat_level": "PLACEHOLDER \u2014 computed by Game-Theoretic module"
}
──────────────────────────────────────────────────────────────────────

⚠️  The threat_level field will be filled by the Game-Theoretic module.
   It uses Bayesian game theory to compute attacker/defender strategies.


---
## Section 14 — Quick Reference Card

```
INGESTION INTERCEPTOR — QUICK REFERENCE
════════════════════════════════════════════════════════════

MODULE POSITION IN SYSTEM:
  [INGESTION INTERCEPTOR]   <-- here (Module 1)
  → Game-Theoretic Threat Estimator
  → Inspection Strategy Selector
  → Multi-Layer Malware Detection (Signature + AI/ML + Sandbox)
  → Metadata Sanitizer
  → Threat Intelligence Correlator
  → Response & Quarantine Manager
  → Logging & Feedback Loop
  → Security Dashboard

INPUT:
  Raw drone submission JSON with payloads and metadata

OUTPUT:
  IngestResult = IngestMetadata + ArtifactRecord[] + warnings/errors

INTERNAL STAGES:
  Stage 1  Validation           validate_submission()     → errors/warnings
  Stage 2  Authentication       Authenticator.authenticate() → AuthResult
  Stage 3  Metadata Extraction  extract_*() functions     → mission/geo/telemetry
  Stage 4  Payload Analysis     analyze_payload()         → security flags
  Stage 5  Checksum Verify      verify_checksum()         → True/False/None
  Stage 6  Artifact Catalog     create_artifact_record()  → ArtifactRecord

SECURITY FLAGS (9 checks):
  encrypted_payload        medium     encryption=True
  nested_archive           medium     container=True
  large_binary             low        size >= 10MB
  suspicious_mime          high       MIME is executable-type
  executable_file          critical   Extension is .exe/.dll/.bat/etc.
  double_extension         high       photo.jpg.exe
  hidden_file              medium     Starts with .
  zero_size_file           medium     0 bytes for video/image
  mime_extension_mismatch  medium     MIME ≠ extension

AUTH OUTCOMES:
  authenticated   Device registered + trusted
  untrusted       Device registered but not trusted
  unknown         Device not in registry (policy=flag)
  rejected        Device revoked or policy=reject

UPLINK COMMANDS:
  QUARANTINE          Mark ingest as quarantined
  RELEASE             Remove quarantine
  REVOKE_DEVICE       Revoke drone trust
  UPDATE_ZONE_RISK    Change zone risk score
  FORCE_RESCAN        Re-analyze submission
  UPDATE_CONFIG       Change config params

KEY CONFIG PARAMS:
  require_signature         Default: False
  max_payload_size_bytes    Default: 500 MB
  unknown_device_policy     Default: "flag"
  checksum_algorithm        Default: sha256
  storage_backend           Default: filesystem

API:
  # Class-based (recommended)
  interceptor = IngestionInterceptor(config, device_registry, zone_risk)
  result = interceptor.process(drone_json)

  # Functional (backward-compatible)
  output_dict = ingestion_interceptor(drone_json, device_registry)

SOURCE FILES:
  ingestion_interceptor/
    config.py             InterceptorConfig
    models.py             DroneSubmission, PayloadEntry, GeoLocation,
                          ArtifactRecord, IngestMetadata, IngestResult
    validator.py          validate_submission()
    authenticator.py      Authenticator, DeviceRegistry, SignatureVerifier
    metadata_extractor.py extract_mission_context(), extract_geo_metadata(),
                          extract_telemetry_summary(), extract_additional_metadata()
    payload_analyzer.py   analyze_payload(), compute_payload_risk_score()
    checksum_verifier.py  compute_file_checksum(), verify_checksum()
    artifact_manager.py   create_artifact_record(), generate_artifact_id()
    uplink.py             UplinkReceiver, UplinkCommandHandler, CommandType
    interceptor.py        IngestionInterceptor, ingestion_interceptor()
```

---

**End of notebook.** You now understand the Ingestion Interceptor from theory to practice.